[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/20_rl_policy_objectives.ipynb)

# 20. RL policy objectives

작은 logits, rewards, advantages만으로 REINFORCE → PPO → DPO → GRPO 계열 objective가 tensor 수준에서 어떻게 달라지는지 본다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. REINFORCE

log-prob에 return을 곱해 policy loss를 만든다.


In [ ]:
logits = torch.tensor([[1.2, 0.3, -0.4]], device=device, requires_grad=True)
action = torch.tensor([0], device=device)
reward = torch.tensor([1.5], device=device)

logp = F.log_softmax(logits, dim=-1).gather(1, action[:, None]).squeeze(1)
loss = -(reward * logp).mean()

print("logp:", logp)
print("loss:", loss.item())
loss.backward()
print("grad:", logits.grad)


In [ ]:
def reinforce_loss(z):
    logp_ = F.log_softmax(z, dim=-1)
    logp_ = logp_.gather(1, action[:, None]).squeeze(1)
    return -(reward * logp_).mean()

_ = profile_call("REINFORCE loss", reinforce_loss, logits.detach())


## 2. Advantage baseline

return에서 baseline을 빼 gradient variance를 줄이는 구조를 본다.


In [ ]:
returns = torch.tensor([2.0, 0.5, 1.0], device=device)
values = torch.tensor([1.2, 0.7, 0.8], device=device)
adv = returns - values

print("advantages:", adv)


In [ ]:
_ = profile_call("advantage", lambda: returns-values)


## 3. PPO clipping

new/old policy ratio를 clip한 surrogate objective와 비교한다.


In [ ]:
old_logp = torch.log(torch.tensor([0.4, 0.3, 0.2], device=device))
new_logp = torch.log(torch.tensor([0.5, 0.2, 0.25], device=device))
adv = torch.tensor([1.0, -0.5, 0.8], device=device)

ratio = (new_logp - old_logp).exp()
clip_ratio = ratio.clamp(0.8, 1.2)

unclipped = ratio * adv
clipped = clip_ratio * adv
ppo_loss = -torch.minimum(unclipped, clipped).mean()

print("ratio:", ratio)
print("clipped ratio:", clip_ratio)
print("PPO loss:", ppo_loss.item())


In [ ]:
def ppo_objective_once():
    ratio_ = (new_logp - old_logp).exp()
    clipped_ = ratio_.clamp(0.8, 1.2)
    return -torch.minimum(ratio_ * adv, clipped_ * adv).mean()

_ = profile_call("PPO clip objective", ppo_objective_once)


## 4. KL regularization

policy와 reference log-prob 차이를 penalty로 넣는다.


In [ ]:
policy_logp = torch.log(torch.tensor([0.5, 0.3, 0.2], device=device))
ref_logp = torch.log(torch.tensor([0.4, 0.35, 0.25], device=device))

approx_kl = (policy_logp - ref_logp).mean()
print("mean log-ratio:", approx_kl.item())


In [ ]:
_ = profile_call("KL log-ratio", lambda: (policy_logp-ref_logp).mean())


## 5. DPO pairwise preference

chosen/rejected log-ratio 차이를 sigmoid objective로 바꾼다.


In [ ]:
pi_chosen = torch.tensor([-0.7, -1.0], device=device)
pi_rejected = torch.tensor([-1.4, -1.2], device=device)
ref_chosen = torch.tensor([-0.9, -1.1], device=device)
ref_rejected = torch.tensor([-1.3, -1.3], device=device)
beta = 0.1

margin = (pi_chosen-pi_rejected) - (ref_chosen-ref_rejected)
dpo_loss = -F.logsigmoid(beta * margin).mean()

print("preference margin:", margin)
print("DPO loss:", dpo_loss.item())


In [ ]:
_ = profile_call("DPO loss", lambda: -F.logsigmoid(beta*margin).mean())


## 6. GRPO group normalization

같은 prompt의 여러 rollout reward를 group 내부에서 normalize한다.


In [ ]:
rewards = torch.tensor([[1.2, 0.2, 0.8, 2.0]], device=device)
mean = rewards.mean(dim=1, keepdim=True)
std = rewards.std(dim=1, keepdim=True, unbiased=False)
group_adv = (rewards - mean) / (std + 1e-6)

print("group rewards:", rewards)
print("group advantages:", group_adv)


In [ ]:
def grpo_normalize_once():
    mean_ = rewards.mean(dim=1, keepdim=True)
    std_ = rewards.std(dim=1, keepdim=True, unbiased=False)
    return (rewards - mean_) / (std_ + 1e-6)

_ = profile_call("GRPO normalize", grpo_normalize_once)


## 7. GRPO-style clipped ratio

group advantage를 PPO형 ratio와 결합한다.


In [ ]:
old = torch.log(torch.tensor([[0.25,0.25,0.25,0.25]], device=device))
new = torch.log(torch.tensor([[0.30,0.20,0.20,0.30]], device=device))
ratio = (new-old).exp()

loss = -torch.minimum(
    ratio * group_adv,
    ratio.clamp(0.8,1.2) * group_adv,
).mean()

print("GRPO-style loss:", loss.item())


In [ ]:
_ = profile_call("GRPO clipped objective", lambda: -torch.minimum(ratio*group_adv,ratio.clamp(.8,1.2)*group_adv).mean())


## References and provenance

**[20.1] REINFORCE**
- 출처: Williams, Simple Statistical Gradient-Following Algorithms
- 이 노트북에서 가져온 부분: score-function policy gradient

**[20.2] PPO**
- 출처: Schulman et al., Proximal Policy Optimization Algorithms
- 이 노트북에서 가져온 부분: clipped policy ratio

**[20.3] DPO**
- 출처: Rafailov et al., Direct Preference Optimization
- 이 노트북에서 가져온 부분: pairwise preference objective

**[20.4] GRPO**
- 출처: DeepSeekMath / DeepSeek-R1 lineage
- 이 노트북에서 가져온 부분: group-relative advantage normalization

**[20.5] DAPO / DrGRPO / GSPO families**
- 출처: recent LLM-RL papers and verl implementations
- 이 노트북에서 가져온 부분: modern variants of group policy optimization; notebook keeps only the common tensor core
